# Llama 4 — mixture-of-experts, vision, and profile-only access

> **Sample code — not for production.** Provided as AWS Content under the AWS
> Customer Agreement; do not use it in production accounts or on production or other
> critical data. Running these cells calls Amazon Bedrock and incurs charges. Full
> disclaimer in the [README](../README.md#disclaimer).

Meta's Llama 4 models are `bedrock-runtime` only and **`INFERENCE_PROFILE` only**:
they reject their bare model ID, so every call goes through the `us.` prefixed
profile. Llama 3.x is still in the catalogue but superseded, so this notebook
covers Llama 4 alone.

| Model | Shape | Input |
|---|---|---|
| `llama4-scout-17b` | mixture-of-experts, smaller | text, image |
| `llama4-maverick-17b` | mixture-of-experts, larger | text, image |

Both are "17b" in the name, which refers to **active** parameters per token, not
total. That is the point of a mixture-of-experts model: a large parameter count
with only a slice active per token. The names are therefore not comparable to a
dense 17b model, and not comparable to each other either.


### Where the helpers come from

The next cell does this:

```python
sys.path.insert(0, "../_shared")
from bedrock import ...
```

`bedrock` is **not** a package from PyPI — it is this collection's own helper
module, [`_shared/bedrock.py`](../_shared/bedrock.py). Every notebook sits one
level down, so `../_shared` puts it on the import path. It exists only to remove
repetition; the notebooks are the teaching material, and nothing in the module is
required to call Bedrock yourself.

What this notebook uses from it:

| Helper | What it does |
|---|---|
| `slide_jpeg` | JPEG bytes of a slide from a public AWS talk, so vision cells have a known answer |
| `converse` | one Converse call; returns `(text, response)` and **never raises** on a service error |
| `converse_tool_uses` | the `toolUse` blocks from a Converse response |
| `endpoints_for` | answers "mantle, runtime, or both" for a model, from the live catalogues |
| `resolve_runtime_id` | turns a model ID into the form Converse will accept, adding the `us.` profile prefix when one is required |
| `runtime_client` | a boto3 `bedrock-runtime` client (Converse, InvokeModel) |
| `runtime_models` | the serverless `bedrock-runtime` catalogue with modalities and inference types |

Two behaviours worth knowing before you read any output below, because several
cells depend on them:

- **`post()` and `converse()` never raise on a service error.** They return the
  status and body so a cell can *show* a 400 rather than stopping the notebook.
  Many cells here deliberately provoke an error to demonstrate a limit.
- **Anything printed from a control-plane response goes through redaction**, since
  this output is committed to a public repository.


In [1]:
import sys

sys.path.insert(0, "../_shared")

from bedrock import (
    SLIDE_CALLOUTS,
    SLIDE_TITLE,
    converse,
    converse_tool_uses,
    endpoints_for,
    resolve_runtime_id,
    keyword_recall,
    runtime_client,
    runtime_models,
    slide_jpeg,
)

REGION = "us-east-1"
SCOUT = "meta.llama4-scout-17b-instruct-v1"
MAVERICK = "meta.llama4-maverick-17b-instruct-v1"

catalogue = runtime_models(REGION)
for model in (SCOUT, MAVERICK):
    entry = catalogue[model]
    print(f"{model}")
    print(f"    input      : {','.join(sorted(entry['in']))}")
    print(f"    inference  : {','.join(sorted(entry['infer']))}")
    print(f"    resolves to: {resolve_runtime_id(model, REGION)}")
    print(f"    endpoints  : {endpoints_for(model, REGION)}")


meta.llama4-scout-17b-instruct-v1
    input      : IMAGE,TEXT
    inference  : INFERENCE_PROFILE


    resolves to: us.meta.llama4-scout-17b-instruct-v1:0


    endpoints  : {'mantle': False, 'runtime': True}
meta.llama4-maverick-17b-instruct-v1
    input      : IMAGE,TEXT
    inference  : INFERENCE_PROFILE
    resolves to: us.meta.llama4-maverick-17b-instruct-v1:0


    endpoints  : {'mantle': False, 'runtime': True}


## 1. The bare ID is rejected

Same category as most of the Claude family. Recognise the error text — it names
on-demand throughput, which sounds like a quota problem and is not.


In [2]:
try:
    runtime_client(REGION).converse(
        modelId=f"{SCOUT}:0",  # bare, no us. prefix
        messages=[{"role": "user", "content": [{"text": "Reply OK"}]}],
        inferenceConfig={"maxTokens": 16},
    )
    print("bare ID accepted")
except Exception as exc:
    print(f"bare ID -> {type(exc).__name__}")
    print(f"    {str(exc)[-150:]}")

text, response = converse(
    SCOUT,
    [{"role": "user", "content": [{"text": "Reply with exactly: OK"}]}],
    max_tokens=16,
    region=REGION,
)
print(f"\nvia profile -> {text.strip()!r}  stop={response.get('stopReason')}")


bare ID -> ValidationException
    t-17b-instruct-v1:0 with on-demand throughput isn’t supported. Retry your request with the ID or ARN of an inference profile that contains this model.



via profile -> 'OK'  stop=end_turn


## 2. Vision, with a known answer

Both models take images. The generated bands give a checkable ground truth, so a
fluent-but-wrong description cannot pass as success.


In [3]:
jpeg = slide_jpeg()
QUESTION = "Read this slide. Give its title, then quote the three green callout lines."
print(f"slide: {len(jpeg)} bytes of JPEG; title is {SLIDE_TITLE!r}\n")

for model in (SCOUT, MAVERICK):
    text, response = converse(
        model,
        [
            {
                "role": "user",
                "content": [
                    {"image": {"format": "jpeg", "source": {"bytes": jpeg}}},
                    {"text": QUESTION},
                ],
            }
        ],
        max_tokens=220,
        region=REGION,
    )
    error = (response.get("error") or {}).get("message")
    if error:
        print(f"{model:<40} ERROR {error[:50]}")
        continue
    hits, total = keyword_recall(text, SLIDE_CALLOUTS)
    title_seen = SLIDE_TITLE.lower() in (text or "").lower()
    print(f"{model:<40} callouts {hits}/{total}  title={title_seen}")
    print(f"{'':<40} {' '.join(text.split())[:80]!r}")

slide: 32687 bytes of JPEG; title is 'Ingestion from database'



meta.llama4-scout-17b-instruct-v1        callouts 3/3  title=True
                                         'The title of the slide is **Ingestion from database**. The three green callout l'


meta.llama4-maverick-17b-instruct-v1     callouts 3/3  title=True
                                         'The title of the slide is "Ingestion from database." The three green callout lin'


## 3. Tool use — and why you must validate every call

Both support tools. Assert the **arguments**, not the fact of a call.

Here is why, recorded from a run while this notebook was being written. Maverick
emitted **three** tool calls for a single question, and the middle one passed back
the tool's own JSON *schema* where the arguments should have been:

```
get_distance_km({'type': 'object', 'properties': {...}, 'required': [...]})
```

A loop that trusted `stopReason: tool_use` and executed every call would have handed
that object to the function. It has not reproduced on every run since — the cell
below usually returns one well-formed call — which is exactly the problem: a
malformed call is rare enough to survive testing and common enough to reach
production.

`stopReason` tells you the model wanted a tool. It says nothing about whether the
arguments are usable. Validate each call against the schema you published before you
execute it, and be ready for more than one call per turn. The cell below checks the
arguments and reports the count.

In [4]:
TOOLS = [
    {
        "toolSpec": {
            "name": "get_distance_km",
            "description": "Great-circle distance between two cities in km",
            "inputSchema": {
                "json": {
                    "type": "object",
                    "properties": {
                        "origin": {"type": "string"},
                        "destination": {"type": "string"},
                    },
                    "required": ["origin", "destination"],
                }
            },
        }
    }
]

for model in (SCOUT, MAVERICK):
    text, response = converse(
        model,
        [
            {
                "role": "user",
                "content": [
                    {"text": "How far is Singapore from Tokyo? Use the tool."}
                ],
            }
        ],
        max_tokens=400,
        tools=TOOLS,
        region=REGION,
    )
    error = (response.get("error") or {}).get("message")
    if error:
        print(f"{model:<40} ERROR {error[:50]}")
        continue
    uses = converse_tool_uses(response)
    print(f"{model:<40} stop={response.get('stopReason')} calls={len(uses)}")
    for use in uses:
        args = use["input"]
        cities = {str(v).lower() for v in args.values()}
        ok = any("singapore" in c for c in cities) and any("tokyo" in c for c in cities)
        print(f"    {use['name']}({args})  {'correct' if ok else 'ARGS WRONG'}")


meta.llama4-scout-17b-instruct-v1        stop=tool_use calls=1
    get_distance_km({'destination': 'Singapore', 'origin': 'Tokyo'})  correct


meta.llama4-maverick-17b-instruct-v1     stop=tool_use calls=3
    get_distance_km({'destination': 'Tokyo', 'origin': 'Singapore'})  correct
    get_distance_km({'type': 'object', 'properties': {'destination': {'type': 'string'}, 'origin': {'type': 'string'}}, 'required': ['origin', 'destination']})  ARGS WRONG
    get_distance_km({'destination': 'Tokyo', 'origin': 'Singapore'})  correct


## 4. Scout or Maverick?

Grade them on a task with a checkable answer. "Bigger is better" is not a
sufficient reason to pay more per token.


In [5]:
TASK = (
    "A train leaves at 09:40 and the journey takes 2 hours 35 minutes. "
    "What time does it arrive? Reply with the time only, 24-hour clock."
)

print(f"{'model':<40} {'tokens':>7}  {'answer':<10} verdict")
print("-" * 76)
for model in (SCOUT, MAVERICK):
    text, response = converse(
        model,
        [{"role": "user", "content": [{"text": TASK}]}],
        max_tokens=200,
        temperature=0.0,
        region=REGION,
    )
    error = (response.get("error") or {}).get("message")
    if error:
        print(f"{model:<40} {'-':>7}  ERROR {error[:26]}")
        continue
    total = response.get("usage", {}).get("totalTokens", 0)
    answer = text.strip().replace("\n", " ")
    verdict = "correct" if "12:15" in answer else "WRONG (expect 12:15)"
    print(f"{model:<40} {total:>7}  {answer[:10]:<10} {verdict}")


model                                     tokens  answer     verdict
----------------------------------------------------------------------------


meta.llama4-scout-17b-instruct-v1             75  12:15      correct


meta.llama4-maverick-17b-instruct-v1          75  12:15      correct


## Takeaways

- **Llama 4 is `bedrock-runtime` only and profile-only.** The bare model ID is
  rejected with an on-demand-throughput error that has nothing to do with quota.
- **"17b" means active parameters, not total.** These are mixture-of-experts
  models, so the number in the name is not comparable to a dense model of the same
  size, nor between Scout and Maverick.
- **Validate every tool call, do not just count them.** Maverick has returned
  three calls for one question with the tool's JSON schema in place of arguments
  (section 3). It does not happen every run, which is what makes it dangerous.
  `stopReason: tool_use` is not a correctness signal.
- **Llama 3.x is still listed but superseded**, and three older Meta and Anthropic
  models are outright blocked as legacy — see `13-amazon-nova/01` section 4 for what
  that refusal looks like.
